## Q-Learning example of Gymnasium Cliff Walking

In [1]:
USE {
    repositories {
        mavenCentral()
        maven("https://central.sonatype.com/repository/maven-snapshots/")
    }
    dependencies {
        implementation("io.github.kotlinrl:integration:0.1.0-SNAPSHOT")
    }
}
%use lets-plot

In [2]:
import io.github.kotlinrl.core.*
import io.github.kotlinrl.integration.gymnasium.*
import io.github.kotlinrl.integration.gymnasium.GymnasiumEnvs.*
import java.io.*

Let's define our hyper-parameters to control training and learning

In [3]:
val maxStepsPerEpisode = 1_000
val trainingEpisodes = 500
val testEpisodes = 15
val initialEpsilon = 1.0
val epsilonDecayRate = 0.9999
val minEpsilon = 0.05
val alpha = 0.6
val gamma = 0.99
val fileName = "CliffWalkingQLearning.csv"

Creating the following
- Env (CliffwalkingEnv = ```Env<Int, Int, Discrete, Discrete>``` based on the Gymnasium
structure)
    - We use a ```TransformState``` wrapper to change the state from ```Int``` space - to a ```MultiDiscrete``` space - which has observations in ```IntArray```.  Essentially we turn the sequence of cells into rows and columns.
    - The ```MultiDiscrete``` space works perfectly for tabular data like the ```QTable```
- ```EpisodeCallback``` to log starting episodes
- ```StateActionListProvider``` to define the list of Actions for any State.  Blackjack only allows
    - Actions
        - 0: Move up
        - 1: Move right
        - 2: Move down
        - 3: Move left
    - State is now ```IntArray``` based on the ```MultiDiscreate``` state space
- The ```QTable``` used to capture training information
    - QLearning updates the ```QTable``` after each ```Trajectory```

In [5]:
val env = TransformState(
    env = gymnasium.make<CliffWalkingEnv>(CliffWalking_v0, render=true),
    transform = { intArrayOf(it / 12, it % 12) },
    observationSpace = MultiDiscrete(4, 12)
)
val episodeLogger = object : EpisodeCallback<IntArray, Int> {
    override fun onEpisodeStart(episode: Int) {
        if (episode > 0 && episode % (trainingEpisodes / 10) == 0) println("Starting episode $episode")
    }
}
val actionListProvider = StateActionListProvider<IntArray, Int> { listOf(0, 1, 2, 3) }

Next we create the training Agent using the QLearning algorithm
- We use an Epsilon Greedy Policy with a decaying epsilon to encourage convergence (experimenting with a constant epsilon would lead to different results) for the exploration factor
- The Epsilon Greedy Policy randomly chooses a number.
    - If it is less than the epsilon value it uses a Random Policy selection
    - Otherwise it uses a Greedy Policy to select the best action from the ```QTable```

The Trainer uses the env and agent with a max steps per episode and trains for the number of training episodes
- The ```qLearningAgent``` function registers the ```QLearning``` algorithm with an ```alpha``` and a ```gamma``` value as a ```TrajectoryObserver``` so that when a ```Trajectory``` completes, the algorithm updates the ```QTable```
- We also register the episode logger
- We then train for the number of training episodes
- When training completes we save the ```QTable``` for later use

In [ ]:
val trainingQtable = QTable(4, 12, 4)
if(File(fileName).exists()) {
    trainingQtable.load(fileName)
    println("QTable loaded from file")
}
val trainingAgent = qLearningAgent(
    id = "training",
    policy = epsilonGreedyPolicy(
        stateActionListProvider = actionListProvider,
        explorationFactor = decayingEpsilon(
            factor = initialEpsilon,
            minFactor = minEpsilon,
            decayRate = epsilonDecayRate
        ),
        qTable = trainingQtable
    ),
    qTable = trainingQtable,
    alpha = alpha,
    gamma = gamma
)
val trainer = episodicTrainer(
    env = env,
    agent = trainingAgent,
    maxStepsPerEpisode = maxStepsPerEpisode,
    callbacks = listOf(
        episodeLogger
    )
)
println("Starting training")
val training = trainer.train(trainingEpisodes)
trainingQtable.save(fileName)

Once training is complete, we create the following
- A new ```QTable``` with the same shape, and load the training data
- A new test ```Agent``` using a ```GreedyPolicy``` against the ```QTable``` with loaded weights
- The Greedy Policy always chooses the best action from the ```QTable```
    - It performs the best action given the state (essentially the agent's row and column)

We then test for the number of testing episodes to compare the episode results (i.e. the average reward achieved)

In [7]:
val testingQtable = QTable(4, 12, 4)
testingQtable.load(fileName)

In [ ]:
val recordEnv = RecordVideo(env = env, folder = "videos/cliff_walking_q_learning", testEpisodes / 4)
val testingAgent = agent(
    id = "testing",
    policy = greedyPolicy(
        qTable = testingQtable
    )
)
val tester = episodicTrainer(
    env = recordEnv,
    agent = testingAgent,
    maxStepsPerEpisode = maxStepsPerEpisode,
    callbacks = listOf(object : EpisodeCallback<IntArray, Int> {
        override fun onEpisodeStart(episode: Int) {
            if (episode > 0 && episode % (testEpisodes / 10) == 0) println("Starting episode $episode")
        }
    })
)
println("Starting testing")
val test = tester.train(testEpisodes)

Comparing the average results between training and testing.

In [9]:
println("Training Results: ${training.episodeRewards.sum() / training.episodeRewards.size}")
println("Test Results: ${test.episodeRewards.sum() / test.episodeRewards.size}")


Training Results: -203.39
Test Results: -13.0


In [10]:
val folder = File(recordEnv.folder)
for(file in folder.listFiles()!!.filter { it.isDirectory }) {
    displayVideo(File(folder, file.name))
}

In [12]:
fun buildQFunctionData(): Map<String, List<Any>> {
    val actionSymbols = mapOf(
        0 to "↑",
        1 to "→",
        2 to "↓",
        3 to "←"
    )

    val x = mutableListOf<Int>()
    val y = mutableListOf<Int>()
    val value = mutableListOf<Double>()
    val action = mutableListOf<String>()

    val shape = testingQtable.shape // shape = [4, 12, 4]
    for (row in 0 until shape[0]) {
        for (col in 0 until shape[1]) {
            val state = intArrayOf(row, col)
            val bestAction = testingQtable.bestAction(state)
            val bestValue = testingQtable.maxValue(state)

            x += col
            y += -row
            value += (bestValue * 100).roundToInt() / 100.0
            action += actionSymbols[bestAction] ?: "?"
        }
    }

    return mapOf(
        "x" to x,
        "y" to y,
        "value" to value,
        "action" to action
    )
}

val data = buildQFunctionData()
val policyPlot = letsPlot(data) +
        geomTile {
            x = "x"
            y = "y"
            fill = "value" // optional for background
        } +
        geomText(
            size = 10,
            color = "black"
        ) {
            x = "x"
            y = "y"
            label = "action"
        } +
        scaleFillGradient(low = "#ffffff", high = "#084594") +
        ggtitle("Policy") +
        ggsize(400, 400)

policyPlot.show()